# Build Predictable Multi-Agent AI Pipelines with Sequential Pattern

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_sequential_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build static sequential agent pipelines where agents execute in a predefined order.

**Pattern:** Fixed sequence of agents with result passing

```
Pipeline: write-review-edit
Input: "Write code to find primes"
  ↓
Step 0: code agent → "def find_primes()..."
  ↓
Step 1: string agent → "Review: 15 lines of code"
  ↓
Step 2: code agent → "def find_primes()... (with docstrings)"
```

**Key concepts:** Predefined pipelines, result passing with placeholders, no LLM planning overhead.

**Why Sequential?**
- ✅ **Predictable**: Know exactly what happens, every time
- ✅ **Fast**: No planning overhead, just execution
- ✅ **Cheap**: Fewer LLM calls (no planner/reflection)
- ✅ **Simple**: Easy to understand, debug, and maintain
- ✅ **Production-ready**: Perfect for known, repeatable workflows

**vs Other Patterns:**
- **Planner**: Dynamic DAG generation → Use when steps unknown upfront
- **ReAct**: Adaptive exploration → Use when need to react to observations
- **Reflection**: Iterative refinement → Use when need high quality outputs
- **Sequential**: Fixed pipeline → Use when workflow is known and repeatable ✨

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [ ]:
view_file("requirements.txt")

In [ ]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [ ]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the sequential workflow. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [ ]:
!python -m workflows.sequential --local --pipeline calculate-explain --input "Calculate 5 factorial"

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [ ]:
!python -m workflows.sequential --pipeline calculate-explain --input "Calculate 5 factorial"

# Code Walkthrough

Let's walk through the code to understand how the sequential workflow is structured and how it works.

**Note:** Sequential uses the same agents and tools as all other workflows. The key difference is the orchestration pattern - predefined fixed sequence vs dynamic planning/adaptation.

## Infrastructure

Sequential uses the same infrastructure as other workflows:

- **Decorators** (`utils/decorators.py`) - Registration for agents and tools
- **Plan Executor** (`utils/plan_executor.py`) - Executes LLM-generated tool plans
- **Agents** (`agents/`) - Same specialist agents (math, string, web_search, code, weather)
- **Tools** (`tools/`) - Same toolsets for each agent

See the [planner tutorial](tutorial_planner_agent.ipynb) for details on these components.

## Sequential Orchestrator - Fixed Pipeline Execution

The sequential pattern executes a predefined sequence of agents with no LLM planning.

**Flow:**
1. **Select Pipeline:** Choose from predefined pipelines (or define your own)
2. **Execute Steps:** Run each agent in exact order
3. **Pass Results:** Use placeholders to inject previous results into next tasks
4. **Return Final:** Last step's output is the final result

**Pipeline Definitions:**
```python
PIPELINES = {
    "write-review-edit": [
        {"agent": "code", "task": "Write Python code for: {input}"},
        {"agent": "string", "task": "Review and count lines: {previous}"},
        {"agent": "code", "task": "Add docstrings to: {previous_0}"}
    ],
    "calculate-explain": [
        {"agent": "math", "task": "{input}"},
        {"agent": "string", "task": "Explain this number: {previous}"}
    ]
}
```

**Smart Placeholders:**
- `{input}` - Original user input (available in all steps)
- `{previous}` - Result from immediately previous step
- `{previous_0}`, `{previous_1}`, etc. - Result from specific step by index

**Example placeholder substitution:**
```
Step 0: task="{input}" → "Calculate 5 factorial"
        result="120"

Step 1: task="Explain {previous}" → "Explain 120"
        result="120 is 5! which means..."

Step 2: task="Use {previous_0} and {previous_1}" 
        → "Use 120 and 120 is 5! which means..."
```

**Key features:**
- Zero LLM planning overhead
- Fully deterministic execution path
- Easy to add custom pipelines
- Error handling stops pipeline on failure

**Agent routing:** Uses `agent_registry` for dynamic dispatch (same as all workflows).

In [ ]:
view_file("workflows/sequential.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.sequential --local \
  --pipeline calculate-explain \
  --input "your input"
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.sequential \
  --pipeline write-review-edit \
  --input "your input"
```
Distributed Flyte cluster, scalable and observable.

**Available Pipelines:**
- `write-review-edit`: Code → Review → Edit (3 steps)
- `calculate-explain`: Math → Explain (2 steps)
- `search-summarize`: Search → Count words (2 steps)
- `weather-analyze`: Weather → Analyze (2 steps)
- `multi-calc`: Math → Math → Math (3 step chain)

**Try these:**

In [ ]:
# Calculate and explain pattern
!python -m workflows.sequential --local --pipeline calculate-explain --input "Calculate 10 factorial"

In [ ]:
# Multi-step calculation chain
!python -m workflows.sequential --local --pipeline multi-calc --input "5 + 3"

In [ ]:
# Search and analyze
!python -m workflows.sequential --local --pipeline search-summarize --input "Python programming language"

## Creating Custom Pipelines

Adding your own pipeline is simple - just add to the `PIPELINES` dictionary in `workflows/sequential.py`:

```python
PIPELINES = {
    # ... existing pipelines ...
    
    "my-custom-pipeline": [
        {"agent": "web_search", "task": "Search for: {input}"},
        {"agent": "string", "task": "Count words in: {previous}"},
        {"agent": "math", "task": "Multiply {previous} by 2"}
    ]
}
```

Then run it:
```bash
python -m workflows.sequential --local --pipeline my-custom-pipeline --input "AI agents"
```

---

## Key Takeaways

**Sequential Pattern:**
- **Fixed sequence:** No planning, just execute predefined steps
- **Result passing:** Smart placeholders inject previous results
- **Deterministic:** Same input → same execution path, every time
- **Efficient:** Minimal LLM overhead, just agent execution

**When to use each pattern:**

| Pattern | Best For | Example Use Case |
|---------|----------|------------------|
| **Sequential** | Known, repeatable workflows | Write → Review → Edit code |
| **Planner** | Unknown steps, maximize parallelism | "Analyze these 3 datasets" |
| **ReAct** | Exploratory, adaptive tasks | "Research and compare options" |
| **Reflection** | High quality requirements | Generate blog post with refinement |

**Production Benefits:**
- **Cost:** Fewer LLM calls = cheaper
- **Speed:** No planning overhead = faster
- **Reliability:** Deterministic = predictable
- **Debuggability:** Fixed sequence = easy to trace

**Common Sequential Patterns:**
- **ETL**: Extract → Transform → Load
- **Content Creation**: Write → Review → Edit → Publish
- **Data Analysis**: Fetch → Process → Analyze → Report
- **Code Quality**: Generate → Lint → Test → Deploy

**Architecture benefits:**
- Same agents/tools work with all patterns
- Type-safe, observable, scalable with Flyte
- Mix and match patterns in larger workflows

**Next steps:**
1. Define your own pipeline for your use case
2. Compare sequential vs planner for same task
3. Combine patterns (e.g., sequential pipeline where one step is a reflection workflow)
4. Build production pipelines with deterministic behavior

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Planner tutorial: [tutorial_planner_agent.ipynb](tutorial_planner_agent.ipynb)
- ReAct tutorial: [tutorial_react_agent.ipynb](tutorial_react_agent.ipynb)
- Reflection tutorial: [tutorial_reflection_agent.ipynb](tutorial_reflection_agent.ipynb)
- Flyte docs: https://docs.flyte.org
- Questions? Join the Flyte community Slack!